# Build Task Bank -- Question Answering

Builds **QATask** objects across five QA datasets covering reading comprehension,
open-domain trivia, multi-hop reasoning, and financial QA.

| Dataset | Domain | Answer type | Hops | Split |
|---------|--------|-------------|------|-------|
| [SQuAD v2](https://huggingface.co/datasets/squad_v2) | Wikipedia | Extractive span | 1 | validation |
| [Natural Questions](https://huggingface.co/datasets/natural_questions) | Wikipedia | Short factual | 1 | validation |
| [TriviaQA rc](https://huggingface.co/datasets/trivia_qa) | Trivia / Wikipedia | Open (aliases) | 1 | validation |
| [MuSiQue](https://huggingface.co/datasets/musique) | Wikipedia | Free-form | 2-4 | validation |
| [FinQA](https://huggingface.co/datasets/ibm/finqa) | Financial reports | Numerical | 1-2 | test |

`context_included=True` for all -- the passage is embedded in `input`.
`answer_extractive=True` only for SQuAD v2.

## 1. Imports & config

In [1]:
import json
import sys
from datetime import date
from pathlib import Path
from typing import Literal

import textstat
from datasets import load_dataset

sys.path.append(str(Path("__file__").resolve().parent.parent))
from models.task import QATask, Task
from utils.pipeline_utils import infer_domain, annotate_pending, _NEEDS_REVIEW

TASK_BANK    = Path("../task_bank")
OUT_SQUAD    = TASK_BANK / "qa_squad2_tasks.jsonl"
OUT_NQ       = TASK_BANK / "qa_nq_tasks.jsonl"
OUT_TRIVIAQA = TASK_BANK / "qa_triviaqa_tasks.jsonl"
OUT_MUSIQUE  = TASK_BANK / "qa_musique_tasks.jsonl"
OUT_FINQA    = TASK_BANK / "qa_finqa_tasks.jsonl"
OUT_COMBINED = TASK_BANK / "qa_combined_tasks.jsonl"
SEED         = 42

N_SQUAD    = 3000
N_NQ       = 3000
N_TRIVIAQA = 3000
N_MUSIQUE  = 3000
N_FINQA    = None   # small dataset -- use all
N_COMBINED = 1000

/home/sristi/.pyenv/versions/venv_langchain_3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load datasets

In [2]:
squad2            = load_dataset("squad_v2",          split="validation")
natural_questions = load_dataset("natural_questions", split="validation")
triviaqa          = load_dataset("trivia_qa",  "rc",  split="validation")
musique           = load_dataset("bdsaglam/musique",  split="validation")
finqa             = load_dataset("ibm/finqa",         split="test")

In [3]:
for name, ds in [("SQuAD v2", squad2), ("NQ", natural_questions),
                 ("TriviaQA", triviaqa), ("MuSiQue", musique), ("FinQA", finqa)]:
    print(f"{name:<12}: {len(ds):>6,} rows | columns: {ds.column_names}")

SQuAD v2    : 11,873 rows | columns: ['id', 'title', 'context', 'question', 'answers']
NQ          :  7,830 rows | columns: ['id', 'document', 'question', 'long_answer_candidates', 'annotations']
TriviaQA    : 17,944 rows | columns: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer']
MuSiQue     :  4,834 rows | columns: ['id', 'paragraphs', 'question', 'question_decomposition', 'answer', 'answer_aliases', 'answerable']
FinQA       :  1,147 rows | columns: ['id', 'pre_text', 'post_text', 'table', 'question', 'answer', 'final_result', 'program_re', 'gold_inds']


In [4]:
import pandas as pd
for name, ds in [("SQuAD v2", squad2), ("NQ", natural_questions),
                 ("TriviaQA", triviaqa), ("MuSiQue", musique), ("FinQA", finqa)]:
    print(f"\n{'='*60}  {name}")
    display(ds.to_pandas().head(2))


============================================================  SQuAD v2


,id,title,context,question,answers
0,56ddde6b9a695914005b9628,Normans,The Normans (Norman: Nourmands; French: Norman...,In what country is Normandy located?,"{'text': ['France', 'France', 'France', 'Franc..."
1,56ddde6b9a695914005b9629,Normans,The Normans (Norman: Nourmands; French: Norman...,When were the Normans in Normandy?,"{'text': ['10th and 11th centuries', 'in the 1..."



============================================================  NQ


,id,document,question,long_answer_candidates,annotations
0,5225754983651766092,"{'html': '<!DOCTYPE html> <HTML class=""client-...",{'text': 'what purpose did seasonal monsoon wi...,"{'end_byte': [44666, 45901, 49618, 50364, 5099...","{'id': ['4323936797498927989', '13037645000009..."
1,6986236841860957647,"{'html': '<!DOCTYPE html> <HTML class=""client-...",{'text': 'where did they film high school musi...,"{'end_byte': [58952, 55477, 55968, 56102, 5623...","{'id': ['4831085488325731996', '11617036854844..."



============================================================  TriviaQA


,question,question_id,question_source,entity_pages,search_results,answer
0,Who was the man behind The Chipmunks?,tc_2,http://www.triviacountry.com/,"{'doc_source': [], 'filename': [], 'title': []...",{'description': ['A struggling songwriter name...,"{'aliases': ['David Seville'], 'normalized_ali..."
1,Which Lloyd Webber musical premiered in the US...,tc_33,http://www.triviacountry.com/,"{'doc_source': ['TagMe'], 'filename': ['Andrew...",{'description': ['The official website for And...,"{'aliases': ['Sunset Blvd', 'West Sunset Boule..."



============================================================  MuSiQue


,id,paragraphs,question,question_decomposition,answer,answer_aliases,answerable
0,2hop__153573_109006,"[{'idx': 0, 'title': 'History of the Internet'...",Who developed the eponymous character from the...,"[{'id': 153573, 'question': 'What series is Mi...",Walt Disney,[],False
1,4hop1__129721_32392_823060_610794,"[{'idx': 0, 'title': 'Zubly Cemetery', 'paragr...",What county is the city that shares a border w...,"[{'id': 129721, 'question': 'In which state is...",Richland County,[],False



============================================================  FinQA


,id,pre_text,post_text,table,question,answer,final_result,program_re,gold_inds
0,ETR/2016/page_23.pdf-2,[entergy corporation and subsidiaries manageme...,[the retail electric price variance is primari...,"[[, amount ( in millions )], [2014 net revenue...",what is the net change in net revenue during 2...,94,94,"subtract(5829, 5735)",[the 2014 net revenue of amount ( in millions ...
1,INTC/2015/page_41.pdf-4,"[item 1b ., unresolved staff comments not appl...",[1 leases on portions of the land used for the...,"[[( square feet in millions ), unitedstates, o...",what percentage of total facilities as measure...,14%,14%,"divide(8.1, 56.0)",[( square feet in millions ) the leased facili...


## 3. Shared helpers

In [ ]:
_FACTOID_Q   = {"who", "what", "when", "where", "which", "whose", "whom"}
_REASONING_Q = {"why", "how", "explain", "describe", "compare", "analyze"}
_STOPWORDS   = {"the", "a", "an", "is", "are", "was", "were", "in",
                "of", "to", "and", "or", "for", "on", "at", "by", "with"}


def assign_difficulty_qa(
    question: str,
    context: str,
    expected: str | list[str],
) -> Literal["easy", "medium", "hard"]:
    """
    Four signals weighted for QA difficulty:

    45%  context length   -- longer context makes it harder to locate
                             the relevant information. Normalised at 300 tokens
                             (typical paragraph length for QA datasets).
    30%  question type    -- factoid starters (who/what/when/where) are easier
                             than reasoning starters (how/why). Also weighted
                             by question length.
    25%  lexical overlap  -- low keyword overlap between question and context
                             means the relevant passage is less signposted.

    Thresholds:  score < 0.30 -> easy  |  < 0.5 -> medium  |  else -> hard
    """
    ctx_tokens = context.split()
    n_ctx      = max(len(ctx_tokens), 1)

    # 1. Context length (normalised at 300 tokens)
    ctx_score  = min(n_ctx / 300.0, 1.0)

    # 2. Question type + length
    q_lower    = question.lower().strip().rstrip("?")
    first_word = q_lower.split()[0] if q_lower.split() else ""
    if first_word in _FACTOID_Q:
        type_score = 0.2
    elif first_word in _REASONING_Q:
        type_score = 0.9
    else:
        type_score = 0.5
    len_score  = min(len(question.split()) / 30.0, 1.0)
    q_score    = 0.65 * type_score + 0.35 * len_score

    # 3. Lexical overlap (low = harder)
    q_kw      = {w for w in q_lower.split() if w not in _STOPWORDS} - _FACTOID_Q - _REASONING_Q
    ctx_words = {w.lower() for w in ctx_tokens}
    overlap   = len(q_kw & ctx_words) / max(len(q_kw), 1)
    ovlp_score= 1.0 - min(overlap, 1.0)

    score = (
        0.45 * ctx_score
        + 0.30 * q_score
        + 0.25 * ovlp_score
    )
    if score < 0.30:
        return "easy"
    if score < 0.5:
        return "medium"
    return "hard"


def shuffle_sample(ds, n, seed=SEED):
    """Deterministic shuffle then take first n rows."""
    shuffled = ds.shuffle(seed=seed)
    return shuffled.select(range(min(n, len(shuffled)))) if n is not None else shuffled

## 4. SQuAD v2

Reading comprehension over Wikipedia passages.
`answer_extractive=True` -- every answer is a verbatim span of the context.
Rows with empty answers (unanswerable questions) are skipped.

In [ ]:
SQUAD_RUBRIC = (
    "Evaluate the model's answer to a reading-comprehension question.\n"
    "The answer must be a verbatim span extracted from the provided context.\n\n"
    "Score on three dimensions (1-5 each):\n"
    "1. Exact match   -- does the answer string exactly match any reference answer?\n"
    "2. Span accuracy -- is the extracted span correctly bounded?\n"
    "3. Faithfulness  -- is the answer supported by the context (no hallucinations)?\n"
    "Return JSON: {\"exact_match\": int, \"span_accuracy\": int, "
    "\"faithfulness\": int, \"overall\": float}"
)


def squad_row_to_task(row: dict, index: int) -> QATask | None:
    answers = row["answers"]["text"]
    if not answers:
        return None          # unanswerable question -- skip
    context  = row["context"]
    question = row["question"]
    return QATask(
        task_id           = f"qa_squad2_val_{index:06d}",
        difficulty        = assign_difficulty_qa(question, context, answers),
        domain            = infer_domain(context),
        input             = f"{context}\n\nQuestion: {question}",
        expected          = answers,
        context_included  = True,
        answer_extractive = True,
        rubric            = SQUAD_RUBRIC,
        metadata          = {
            "source"     : "squad_v2",
            "split"      : "validation",
            "title"      : row.get("title", ""),
            "original_id": row.get("id", index),
            "date_added" : str(date.today()),
        },
    )

skipped_tasks = []
skipped_count = 0
squad_sample = shuffle_sample(squad2, N_SQUAD)
squad_tasks, skipped = [], 0
for i in range(len(squad_sample)):
    t = squad_row_to_task(squad_sample[i], i)
    if t is None: 
        skipped_count += 1
        skipped_tasks.append(squad_sample[i])
    else: squad_tasks.append(t)
print(f"SQuAD v2: {len(squad_tasks)} tasks  ({skipped_count} skipped -- unanswerable)")

In [ ]:
skipped_tasks[2]

In [ ]:
squad_tasks[0]

## 5. Natural Questions

Real Google search queries answered from Wikipedia.
Context is extracted from the annotated **long-answer** token span (non-HTML tokens only).
Rows without a short answer or long answer span are skipped.

In [ ]:
NQ_RUBRIC = (
    "Evaluate the model's answer to a factual question from Natural Questions.\n"
    "Answers are short factual phrases drawn from Wikipedia.\n\n"
    "Score on three dimensions (1-5 each):\n"
    "1. Correctness  -- does the answer match the reference short answer?\n"
    "2. Conciseness  -- is the answer appropriately brief?\n"
    "3. Faithfulness -- is the answer supported by the provided passage?\n"
    "Return JSON: {\"correctness\": int, \"conciseness\": int, "
    "\"faithfulness\": int, \"overall\": float}"
)


def _extract_nq_context(row: dict) -> str | None:
    """Extract long-answer span text from NQ document tokens (skips HTML tokens)."""
    long_ans = row["annotations"]["long_answer"][0]
    if long_ans["start_token"] < 0:
        return None
    tokens   = row["document"]["tokens"]
    start, end = long_ans["start_token"], long_ans["end_token"]
    tok_texts  = tokens["token"][start:end]
    is_html    = tokens["is_html"][start:end]
    text_parts = [tok for tok, html in zip(tok_texts, is_html) if not html]
    return " ".join(text_parts).strip() or None


def nq_row_to_task(row: dict, index: int) -> QATask | None:
    short_answers = row["annotations"]["short_answers"][0]
    expected = [t for t in short_answers["text"] if t]
    if not expected:
        return None
    context = _extract_nq_context(row)
    if not context:
        return None
    question = row["question"]["text"]
    return QATask(
        task_id           = f"qa_nq_val_{index:06d}",
        difficulty        = assign_difficulty_qa(question, context, expected),
        domain            = infer_domain(context),
        input             = f"{context}\n\nQuestion: {question}",
        expected          = expected,
        context_included  = True,
        answer_extractive = False,
        rubric            = NQ_RUBRIC,
        metadata          = {
            "source"     : "natural_questions",
            "split"      : "validation",
            "original_id": row.get("id", index),
            "date_added" : str(date.today()),
        },
    )


nq_sample = shuffle_sample(natural_questions, N_NQ)
nq_tasks, skipped = [], 0
for i in range(len(nq_sample)):
    t = nq_row_to_task(nq_sample[i], i)
    if t is None: skipped += 1
    else: nq_tasks.append(t)
print(f"Natural Questions: {len(nq_tasks)} tasks  ({skipped} skipped -- no short/long answer)")

In [ ]:
print(nq_sample[1])

## 6. TriviaQA (rc)

Trivia questions paired with Wikipedia evidence passages.
`expected` includes the canonical answer plus all valid aliases.

In [ ]:
TRIVIAQA_RUBRIC = (
    "Evaluate the model's answer to a trivia question grounded in a Wikipedia passage.\n"
    "The expected field lists all acceptable answer forms (canonical + aliases).\n\n"
    "Score on three dimensions (1-5 each):\n"
    "1. Correctness  -- does the answer match any of the acceptable answer strings?\n"
    "2. Conciseness  -- is the answer a focused phrase, not a full sentence?\n"
    "3. Grounding    -- is the answer supported by the provided passage?\n"
    "Return JSON: {\"correctness\": int, \"conciseness\": int, "
    "\"grounding\": int, \"overall\": float}"
)


def triviaqa_row_to_task(row: dict, index: int) -> QATask | None:
    wiki_contexts = row.get("entity_pages", {}).get("wiki_context", [])
    if not wiki_contexts:
        return None
    context  = wiki_contexts[0][:3000]     # cap to first 3 000 chars
    question = row["question"]
    answer   = row["answer"]
    expected = [answer["value"]] + list(answer.get("aliases", []))
    return QATask(
        task_id           = f"qa_triviaqa_val_{index:06d}",
        difficulty        = assign_difficulty_qa(question, context, expected),
        domain            = infer_domain(context),
        input             = f"{context}\n\nQuestion: {question}",
        expected          = expected,
        context_included  = True,
        answer_extractive = False,
        rubric            = TRIVIAQA_RUBRIC,
        metadata          = {
            "source"     : "trivia_qa",
            "config"     : "rc",
            "split"      : "validation",
            "original_id": row.get("question_id", index),
            "date_added" : str(date.today()),
        },
    )


tqa_sample = shuffle_sample(triviaqa, N_TRIVIAQA)
tqa_tasks, skipped = [], 0
for i in range(len(tqa_sample)):
    t = triviaqa_row_to_task(tqa_sample[i], i)
    if t is None: skipped += 1
    else: tqa_tasks.append(t)
print(f"TriviaQA: {len(tqa_tasks)} tasks  ({skipped} skipped -- no wiki context)")

## 7. MuSiQue

Multi-hop Wikipedia QA requiring 2-4 reasoning steps.
Context is built from **supporting paragraphs only** (`is_supporting=True`).
Unanswerable questions are skipped.

In [ ]:
MUSIQUE_RUBRIC = (
    "Evaluate the model's answer to a multi-hop reasoning question.\n"
    "The answer requires combining information from multiple paragraphs.\n\n"
    "Score on three dimensions (1-5 each):\n"
    "1. Correctness  -- does the answer match the reference or its aliases?\n"
    "2. Reasoning    -- does the answer demonstrate multi-hop inference "
    "(not surface-level matching)?\n"
    "3. Faithfulness -- is every reasoning step supported by the provided "
    "paragraphs (no hallucinated bridging facts)?\n"
    "Return JSON: {\"correctness\": int, \"reasoning\": int, "
    "\"faithfulness\": int, \"overall\": float}"
)


def musique_row_to_task(row: dict, index: int) -> QATask | None:
    if not row.get("answerable", True):
        return None
    supporting = [p for p in row["paragraphs"] if p["is_supporting"]]
    if not supporting:
        return None
    context  = "\n\n".join(f"[{p['title']}]\n{p['paragraph_text']}" for p in supporting)
    question = row["question"]
    expected = [row["answer"]] + list(row.get("answer_aliases", []))
    return QATask(
        task_id           = f"qa_musique_val_{index:06d}",
        difficulty        = assign_difficulty_qa(question, context, expected),
        domain            = infer_domain(context),
        input             = f"{context}\n\nQuestion: {question}",
        expected          = expected,
        context_included  = True,
        answer_extractive = False,
        rubric            = MUSIQUE_RUBRIC,
        metadata          = {
            "source"     : "musique",
            "split"      : "validation",
            "original_id": row.get("id", index),
            "num_hops"   : len(row.get("question_decomposition", [])),
            "date_added" : str(date.today()),
        },
    )


musique_sample = shuffle_sample(musique, N_MUSIQUE)
musique_tasks, skipped = [], 0
for i in range(len(musique_sample)):
    t = musique_row_to_task(musique_sample[i], i)
    if t is None: skipped += 1
    else: musique_tasks.append(t)
print(f"MuSiQue: {len(musique_tasks)} tasks  ({skipped} skipped -- unanswerable)")

## 8. FinQA

Financial QA over earnings reports requiring numerical reasoning over tables and text.
Context = text paragraphs + formatted table.
`expected` is the executed numerical answer (`exe_ans`). Domain fixed to `business`.

In [ ]:
FINQA_RUBRIC = (
    "Evaluate the model's answer to a financial reasoning question.\n"
    "The question requires numerical computation over the provided table and text.\n\n"
    "Score on three dimensions (1-5 each):\n"
    "1. Numerical accuracy -- is the final numeric value correct "
    "(within rounding tolerance)?\n"
    "2. Reasoning steps    -- are the intermediate arithmetic operations correct?\n"
    "3. Unit / format      -- does the answer use the correct unit and format "
    "(%%, $, ratio, etc.)?\n"
    "Return JSON: {\"numerical_accuracy\": int, \"reasoning_steps\": int, "
    "\"unit_format\": int, \"overall\": float}"
)


def _format_table(table) -> str:
    if not table:
        return ""
    rows = [" | ".join(str(cell).strip() for cell in row) for row in table]
    return "Table:\n" + "\n".join(rows)


def finqa_row_to_task(row: dict, index: int) -> QATask | None:
    para_text  = "\n".join(p.strip() for p in (row.get("paragraphs") or []) if p.strip())
    table_text = _format_table(row.get("table", []))
    context    = f"{para_text}\n\n{table_text}".strip()
    if not context:
        return None
    question = row["question"]
    exe_ans  = row.get("exe_ans")
    expected = str(exe_ans) if exe_ans is not None else str(row.get("answer", ""))
    if not expected:
        return None
    return QATask(
        task_id           = f"qa_finqa_test_{index:06d}",
        difficulty        = assign_difficulty_qa(question, context, expected),
        domain            = "business",
        input             = f"{context}\n\nQuestion: {question}",
        expected          = expected,
        context_included  = True,
        answer_extractive = False,
        rubric            = FINQA_RUBRIC,
        metadata          = {
            "source"     : "ibm/finqa",
            "split"      : "test",
            "original_id": row.get("id", index),
            "date_added" : str(date.today()),
        },
    )


finqa_sample = shuffle_sample(finqa, N_FINQA)
finqa_tasks, skipped = [], 0
for i in range(len(finqa_sample)):
    t = finqa_row_to_task(finqa_sample[i], i)
    if t is None: skipped += 1
    else: finqa_tasks.append(t)
print(f"FinQA: {len(finqa_tasks)} tasks  ({skipped} skipped -- no context or answer)")

## 9. Human annotation pass

In [ ]:
all_dicts = {
    "SQuAD v2" : [t.model_dump() for t in squad_tasks],
    "NQ"       : [t.model_dump() for t in nq_tasks],
    "TriviaQA" : [t.model_dump() for t in tqa_tasks],
    "MuSiQue"  : [t.model_dump() for t in musique_tasks],
    "FinQA"    : [t.model_dump() for t in finqa_tasks],
}

for name, tasks in all_dicts.items():
    pending = sum(1 for t in tasks if t["domain"] == _NEEDS_REVIEW)
    if pending:
        print(f"{name}: {pending} tasks need annotation")
        annotate_pending(tasks)
    else:
        print(f"{name}: all domains auto-tagged")

## 10. Difficulty distribution

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

difficulties = ["easy", "medium", "hard"]
bar_colors   = {"easy": "#4caf50", "medium": "#ff9800", "hard": "#f44336"}
names        = list(all_dicts.keys())
x            = np.arange(len(names))
width        = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for j, diff in enumerate(difficulties):
    totals = [len(all_dicts[n]) for n in names]
    pcts   = [
        sum(1 for t in all_dicts[n] if t["difficulty"] == diff) / totals[i] * 100
        for i, n in enumerate(names)
    ]
    bars = ax.bar(x + j * width, pcts, width, label=diff, color=bar_colors[diff])
    ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel("Percentage of tasks (%)")
ax.set_ylim(0, 120)
ax.set_title("Difficulty distribution by QA dataset")
ax.legend(title="Difficulty")
plt.tight_layout()
plt.show()

print(f"\n{'Dataset':<12} {'easy':>6} {'medium':>8} {'hard':>6} {'total':>7}")
print("-" * 42)
for name, tasks in all_dicts.items():
    c = Counter(t["difficulty"] for t in tasks)
    print(f"{name:<12} {c['easy']:>6} {c['medium']:>8} {c['hard']:>6} {len(tasks):>7}")

## 11. Save to JSONL

In [ ]:
import random as _random

TASK_BANK.mkdir(parents=True, exist_ok=True)

save_map = [
    (OUT_SQUAD,    all_dicts["SQuAD v2"]),
    (OUT_NQ,       all_dicts["NQ"]),
    (OUT_TRIVIAQA, all_dicts["TriviaQA"]),
    (OUT_MUSIQUE,  all_dicts["MuSiQue"]),
    (OUT_FINQA,    all_dicts["FinQA"]),
]

for out_path, tasks in save_map:
    with out_path.open("w", encoding="utf-8") as f:
        for task in tasks:
            f.write(json.dumps(task, ensure_ascii=False) + "\n")
    print(f"Saved {len(tasks):>4} tasks -> {out_path.name}")

_random.seed(SEED)
combined = [
    task
    for _, tasks in save_map
    for task in _random.sample(
        tasks, len(tasks) if N_COMBINED is None else min(N_COMBINED, len(tasks))
    )
]
_random.shuffle(combined)
with OUT_COMBINED.open("w", encoding="utf-8") as f:
    for task in combined:
        f.write(json.dumps(task, ensure_ascii=False) + "\n")
print(f"\nSaved {len(combined):>4} tasks -> {OUT_COMBINED.name}  (combined sample)")

## 12. Reload & validate round-trip

In [ ]:
from pydantic import TypeAdapter

adapter = TypeAdapter(Task)

for out_path, _ in [*save_map, (OUT_COMBINED, None)]:
    loaded = []
    with out_path.open(encoding="utf-8") as f:
        for line in f:
            loaded.append(adapter.validate_json(line))
    ok = all(isinstance(t, QATask) for t in loaded)
    print(f"{out_path.name:<35} {len(loaded):>4} tasks  all QATask: {ok}")